# Block 1 : Envaironment & Reproduciblity

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO
from pathlib import Path
import numpy as np 
import random
import json
import sys
import os 

# Reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Ensure deterministic behavior for consistent mAP testing

torch.backends.cudnn.deterministic = True
print(f"Environment ready with seed {SEED}")

# Block 2 : Configuration & Hyperparameters 

In [ ]:
# Hyperparameters Configuration

def get_training_config():
    return {
        "imgsz" : 640,
        "batch" : 16,
        "optimizer" : "AdamW",
        "lr0" : 0.001,
        "lrf" : 0.01,
        "weight_decay" : 0.0005,
        "patience" : 50,
        "mosaic" : 1.0,
        "mixup" : 0.15,
        "close_mosaic" : 10  
        
    }

# Block 3 : The Architecture & Pretrained Logic

In [ ]:
# Loading the Model


# Load model
model = YOLO('yolo11n.pt')

# Helper function to control layers trainability

def freez_layers(model, trainable_indices):
    for name, param in model.model.named_parameters():
        parts = name.split(".")
        if len(parts) >= 2 and parts[9].isdigit():
            idx = int(parts[9])
            param.requires_grad = idx in trainable_indices


# Block 4 : Feature Extraction (Head training)

In [ ]:
# Stage 1 : Train the head

freez_layers(model, trainable_indices = [23])

config = get_training_config()
model.train(
    data = "datasets/NEU-DET/yolo_preprocessed/dataset.yaml",
    epochs = 50,
    name = "neu_det_stage1",
    **config
)

# Block 5 : Stage 2 (Fine Tuning)

In [ ]:
# Unfreeze everything and lower learning rate

for param in model.model.parameters():
    param.requires_grad = True

config["lr0"] = 0.0001  # 10x lower learning rate to avoid destroying pretrained weights
model.train(
    data = "datasets/NEU-DET/yolo_preprocessed/dataset.yaml",
    epochs = 200,
    name = "neu_det_fine_tune",
    **config
)

# Block 6 : Regularization & Augmentation Awareness

during stage 2 our odel is at high risk of Overfitting, we use mixup and weight_decay defined in our config to ensure the model generalizes to all steel surfaces

# Block 7 : Evaluation & Metrics 

once training finishes, we must evaluate using mAP@.5 and mAP@.5:.95

In [ ]:
# Professional evaluation

results = model.val()
print (f"final mAP@0.5 : {results.box.map50}")
print (f"final mAP@0.5:0.95 : {results.box.map}")